In [0]:

storage_account_name = "mempoolspaceacc"
storage_account_key = dbutils.secrets.get(scope = 'vaultmempool', key = 'membool-access-key')

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)
display(dbutils.fs.ls(f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/"))

In [0]:
bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/mempool_delta"
stream_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/mempoolspace-events/mempool_realtime_streamig"

df_avro = (spark.read.format("avro")
           .option("recursiveFileLookup", "true")
           .option("ignoreExtension", "true")
           .load(stream_path))

df_avro.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

df_silver = spark.read.format("delta").load(bronze_path)
display(df_silver)

In [0]:
print("Total rec in container:", df_avro.count())

In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType

df_decoded = df_avro.withColumn("body_str", col("Body").cast("string"))
sample_json = df_decoded.select("body_str").first()["body_str"]
json_schema = schema_of_json(sample_json)

df_cleaned = df_decoded.withColumn("data", from_json(col("body_str"), json_schema)) \
                       .withColumn("sequence_number", col("SequenceNumber")) \
                       .withColumn("enqueued_time", col("EnqueuedTimeUtc")) \
                       .select("sequence_number", "enqueued_time", "data.*")

display(df_cleaned)

In [0]:
from pyspark.sql.functions import col\
    , explode_outer

df_exploded = df_cleaned.withColumn("block_item", explode_outer(col("data")))

df_flattened = df_exploded.select(
    col("sequence_number")\
    , col("enqueued_time")\
    , col("block_item.blockSize").alias("block_size")\
    , col("block_item.blockVSize").alias("block_vsize")\
    , col("block_item.feeRange").alias("fee_range")\
    , col("block_item.medianFee").alias("median_fee")\
    , col("block_item.nTx").alias("n_tx")\
    , col("block_item.totalFees").alias("total_fees")\
    , col("block_item").alias("raw_block_object")
)

display(df_flattened)

In [0]:
from pyspark.sql.functions import col\
    , explode_outer

df_exploded = df_cleaned.withColumn("block_item", explode_outer(col("data")))

df_final = df_exploded.select(
    col("sequence_number")\
    , col("enqueued_time")\
    , col("block_item.blockSize").alias("block_size")\
    , col("block_item.blockVSize").alias("block_vsize")\
    , col("block_item.medianFee").alias("median_fee")\
    , col("block_item.nTx").alias("n_tx")\
    , col("block_item.totalFees").alias("total_fees")\
    , col("block_item.feeRange")[0].alias("fee_p10")\
    , col("block_item.feeRange")[1].alias("fee_p25")\
    , col("block_item.feeRange")[2].alias("fee_p50")\
    , col("block_item.feeRange")[3].alias("fee_p75")\
    , col("block_item.feeRange")[4].alias("fee_p90")\
    , col("block_item.feeRange")[5].alias("fee_p95")\
    , col("block_item.feeRange")[6].alias("fee_p99")\
)

display(df_final)

In [0]:
output_stream_path = "abfss://bronze@mempoolspaceacc.dfs.core.windows.net/Mempool_parsing"
df_final.write.mode("overwrite").parquet(output_stream_path)
print("Data written successfully to ADLS Gen2!")